# Runtime — Unified Execution with Observability and Optional Backends

**Multigen SDK — Notebook 18**

---

Every previous notebook focused on **what** to run: Chains, Parallels, Graphs, StateMachines, message buses.

This notebook focuses on **how to run them** — using the `Runtime` class, which provides:

1. **Unified execution entry point** — one API for all workflow types
2. **Event emission** — every execution step emits structured events
3. **Simulator integration** — events stream to a visual dashboard in real-time
4. **Backend switching** — swap in Temporal (durable) or Kafka (distributed) without changing your workflow code
5. **Statistics tracking** — success rates, latency, workflow counts
6. **Process-level singleton** — `get_runtime()` for shared runtime across your codebase

---

### Runtime Architecture

```
Your Workflow Code
       │
       │ await rt.run_chain(chain, input)
       ▼
┌─────────────────────────────┐
│        Runtime              │
│  ┌───────────────────────┐  │
│  │   Workflow Executor   │  │
│  └────────┬──────────────┘  │
│           │ executes        │
│  ┌────────▼──────────────┐  │
│  │  Chain / Parallel /   │  │
│  │  Graph / StateMachine │  │
│  └───────────────────────┘  │
│           │ emits events    │
│  ┌────────▼──────────────┐  │
│  │   Event Emitter       │──────► Simulator UI (POST /api/v1/events)
│  └───────────────────────┘  │
│           │                 │
│  ┌────────▼──────────────┐  │
│  │   Stats Collector     │  │
│  └───────────────────────┘  │
└─────────────────────────────┘
```

## Section 1: What Runtime Does

### Problem: Scattered Execution

Without Runtime, you'd call each workflow type directly:
```python
await chain.run(input)           # Different method
await parallel.run(input)        # Different method  
await graph.run(input)           # Different method
await state_machine.run(input)   # Different method
```

Each call is isolated — no unified logging, no observability, no stats.

### Solution: Unified Runtime Entry Point

With Runtime:
```python
rt = Runtime()
await rt.run_chain(chain, input)           # Consistent API
await rt.run_parallel(parallel, input)     # Consistent API
await rt.run_graph(graph, input)           # Consistent API
await rt.run_state_machine(sm, input)      # Consistent API
```

Every call:
- Gets a unique `workflow_id` (for tracing)
- Emits structured events before, during, and after execution
- Updates stats counters
- Optionally streams to the Simulator UI or a backend service

### Key Benefits

| Benefit | Description |
|---|---|
| **Observability** | Every step emits events with sender, content, trace_id |
| **Tracing** | All events from one workflow share a `trace_id` |
| **Stats** | Automatic success rate, latency, error tracking |
| **Backend Agnostic** | Swap execution backend without changing workflow code |
| **Simulator** | Real-time visual dashboard for development debugging |

In [ ]:
# SDK Imports
import asyncio
import time
from multigen import Runtime, get_runtime, Chain, Parallel, FunctionAgent

print("Imports loaded successfully.")
print("Ready to explore the Runtime execution layer.")

## Section 2: Creating a Runtime

Three common ways to create a Runtime, depending on your use case:

1. **`Runtime()`** — basic runtime, local execution, minimal config
2. **`Runtime(simulator_url=...)`** — stream events to the Simulator UI
3. **`Runtime(max_concurrency=20, trace=True)`** — production config with full tracing

Constructor parameters:

| Parameter | Type | Default | Description |
|---|---|---|---|
| `simulator_url` | str | None | URL for Simulator event streaming |
| `max_concurrency` | int | 10 | Max concurrent workflow executions |
| `trace` | bool | False | Enable full distributed tracing |
| `timeout` | float | 300.0 | Default timeout per workflow (seconds) |
| `retry_on_failure` | bool | False | Auto-retry failed workflows once |
| `log_level` | str | "INFO" | Logging verbosity |

In [ ]:
# --- Creating Runtime instances ---

# Option 1: Basic runtime — no simulator, local execution
rt_basic = Runtime()
print("Basic Runtime:")
print(f"  type:             {type(rt_basic).__name__}")
print(f"  simulator_url:    {rt_basic.simulator_url!r}")
print(f"  max_concurrency:  {rt_basic.max_concurrency}")
print(f"  trace:            {rt_basic.trace}")
print(f"  timeout:          {rt_basic.timeout}s")
print()

# Option 2: Runtime with Simulator integration
# The Simulator is a local web server (run: python -m multigen.simulator)
# Events from this runtime will stream to the dashboard at http://localhost:8080
rt_with_simulator = Runtime(
    simulator_url="http://localhost:8080"
)
print("Runtime with Simulator integration:")
print(f"  simulator_url:    {rt_with_simulator.simulator_url}")
print(f"  Events will be POSTed to: {rt_with_simulator.simulator_url}/api/v1/events")
print()

# Option 3: Production runtime with full config
rt_production = Runtime(
    simulator_url="http://localhost:8080",  # Simulator for dev
    max_concurrency=20,   # Allow 20 concurrent workflow runs
    trace=True,           # Enable distributed tracing (OpenTelemetry-compatible)
    timeout=600.0,        # 10 minute timeout for long workflows
    retry_on_failure=True, # Auto-retry once if a workflow fails
    log_level="DEBUG"     # Verbose logging
)
print("Production Runtime:")
print(f"  simulator_url:    {rt_production.simulator_url}")
print(f"  max_concurrency:  {rt_production.max_concurrency}")
print(f"  trace:            {rt_production.trace}")
print(f"  timeout:          {rt_production.timeout}s")
print(f"  retry_on_failure: {rt_production.retry_on_failure}")
print(f"  log_level:        {rt_production.log_level}")

# Use the basic runtime for the rest of this notebook
rt = rt_basic
print()
print("Using rt_basic for demos below.")

## Section 3: rt.run_chain()

Run any `Chain` through the runtime. The `workflow_id` parameter lets you group events and stats under a meaningful identifier.

If you don't provide a `workflow_id`, the runtime generates a UUID automatically.

In [ ]:
# --- Setup: Define some reusable test agents ---

async def fetch_agent_fn(query, context=None):
    """Simulates fetching data from an external source."""
    await asyncio.sleep(0.02)
    return {"query": query, "results": [f"result_{i}" for i in range(5)], "source": "database"}

async def analyze_agent_fn(data, context=None):
    """Analyzes the fetched data."""
    await asyncio.sleep(0.015)
    results = data.get("results", []) if isinstance(data, dict) else []
    return {
        "analyzed": True,
        "insights": [f"Insight from {r}" for r in results[:3]],
        "count": len(results)
    }

async def format_agent_fn(analysis, context=None):
    """Formats the analysis into a presentable report."""
    await asyncio.sleep(0.01)
    insights = analysis.get("insights", []) if isinstance(analysis, dict) else []
    return {
        "report": "\n".join(insights),
        "format": "markdown",
        "sections": len(insights)
    }

fetch_agent  = FunctionAgent(fetch_agent_fn,   name="FetchAgent")
analyze_agent = FunctionAgent(analyze_agent_fn, name="AnalyzeAgent")
format_agent  = FunctionAgent(format_agent_fn,  name="FormatAgent")

# Build a chain
analysis_chain = Chain(
    fetch_agent,
    analyze_agent,
    format_agent,
    name="AnalysisChain"
)

print("Agents and chain created.")
print(f"  Chain: {analysis_chain.name} ({len(analysis_chain.agents)} agents)")
print()

# --- Run the chain through the runtime ---
async def demo_run_chain():
    print("=" * 60)
    print("rt.run_chain() — Running AnalysisChain via Runtime")
    print("=" * 60)
    
    # Without a workflow_id — runtime generates one
    print("\nRun 1: No workflow_id (auto-generated UUID):")
    result1 = await rt.run_chain(
        chain=analysis_chain,
        input_data="What are the trends in Q4 sales data?"
    )
    print(f"  result: {result1}")
    print(f"  workflow_id (auto): {result1.workflow_id}")
    
    # With a named workflow_id — useful for correlation in logs
    print("\nRun 2: Named workflow_id:")
    result2 = await rt.run_chain(
        chain=analysis_chain,
        input_data="Q4 sales analysis request",
        workflow_id="q4-sales-analysis-2025"
    )
    print(f"  result: {result2}")
    print(f"  workflow_id (named): {result2.workflow_id}")
    print(f"  duration: {result2.duration_ms:.1f}ms")
    print(f"  status:   {result2.status}")

asyncio.run(demo_run_chain())

## Section 4: rt.run_parallel()

Run any `Parallel` workflow through the runtime. All agents in the Parallel execute concurrently, and the runtime tracks the combined execution as a single workflow.

In [ ]:
# --- rt.run_parallel() ---

from multigen import Parallel

async def sentiment_fn(text, context=None):
    await asyncio.sleep(0.03)
    import random
    return {"sentiment": random.choice(["positive", "neutral", "negative"]), "score": round(random.uniform(-1, 1), 2)}

async def keywords_fn(text, context=None):
    await asyncio.sleep(0.02)
    words = str(text).split()[:5] if text else []
    return {"keywords": words, "count": len(words)}

async def readability_fn(text, context=None):
    await asyncio.sleep(0.025)
    import random
    return {"readability_score": round(random.uniform(60, 90), 1), "grade_level": random.randint(8, 14)}

# Create agents
sentiment_agent    = FunctionAgent(sentiment_fn,    name="SentimentAgent")
keywords_agent     = FunctionAgent(keywords_fn,     name="KeywordsAgent")
readability_agent  = FunctionAgent(readability_fn,  name="ReadabilityAgent")

# Build a parallel workflow — all 3 agents run concurrently
text_analysis_parallel = Parallel(
    sentiment_agent,
    keywords_agent,
    readability_agent,
    name="TextAnalysisParallel"
)

async def demo_run_parallel():
    print("=" * 60)
    print("rt.run_parallel() — 3 concurrent agents")
    print("=" * 60)
    
    text = "The quarterly earnings report shows significant growth in emerging markets."
    
    start = time.time()
    result = await rt.run_parallel(
        parallel=text_analysis_parallel,
        input_data=text,
        workflow_id="text-analysis-parallel-001"
    )
    elapsed = time.time() - start
    
    print(f"\nInput: '{text[:60]}...'")
    print(f"\nResults (3 agents ran concurrently):")
    print(f"  sentiment:   {result.output.get('SentimentAgent')}")
    print(f"  keywords:    {result.output.get('KeywordsAgent')}")
    print(f"  readability: {result.output.get('ReadabilityAgent')}")
    print()
    print(f"Wall time:     {elapsed*1000:.0f}ms")
    print(f"Expected:      ~30ms (slowest agent), NOT ~75ms (sequential sum)")
    print(f"workflow_id:   {result.workflow_id}")
    print(f"duration_ms:   {result.duration_ms:.1f}ms")
    print(f"status:        {result.status}")

asyncio.run(demo_run_parallel())

## Section 5: rt.run_graph()

Run a `Graph` workflow (with conditional routing) through the runtime. The graph's execution path is determined at runtime based on agent outputs, and all events are traced.

In [ ]:
# --- rt.run_graph() ---

from multigen import Graph

# Define agents for a conditional routing graph
async def classifier_fn(text, context=None):
    """Classifies input and determines routing."""
    await asyncio.sleep(0.01)
    import random
    category = random.choice(["technical", "business", "legal"])
    return {"text": text, "category": category, "confidence": round(random.uniform(0.7, 0.99), 2)}

async def technical_handler_fn(data, context=None):
    await asyncio.sleep(0.02)
    return {"handled_by": "TechnicalTeam", "ticket_id": "TECH-001", **data}

async def business_handler_fn(data, context=None):
    await asyncio.sleep(0.02)
    return {"handled_by": "BusinessTeam", "priority": "high", **data}

async def legal_handler_fn(data, context=None):
    await asyncio.sleep(0.02)
    return {"handled_by": "LegalTeam", "review_required": True, **data}

classifier_agent       = FunctionAgent(classifier_fn,        name="ClassifierAgent")
technical_handler      = FunctionAgent(technical_handler_fn,  name="TechnicalHandler")
business_handler       = FunctionAgent(business_handler_fn,   name="BusinessHandler")
legal_handler          = FunctionAgent(legal_handler_fn,      name="LegalHandler")

# Build graph with conditional routing
routing_graph = Graph(name="RequestRoutingGraph")
routing_graph.add_node("classify",  classifier_agent)
routing_graph.add_node("technical", technical_handler)
routing_graph.add_node("business",  business_handler)
routing_graph.add_node("legal",     legal_handler)

routing_graph.add_edge(
    "classify", "technical",
    condition=lambda out: out.get("category") == "technical"
)
routing_graph.add_edge(
    "classify", "business",
    condition=lambda out: out.get("category") == "business"
)
routing_graph.add_edge(
    "classify", "legal",
    condition=lambda out: out.get("category") == "legal"
)

routing_graph.set_entry("classify")

async def demo_run_graph():
    print("=" * 60)
    print("rt.run_graph() — Conditional routing graph")
    print("=" * 60)
    
    # Run 3 requests — each will be classified differently
    requests = [
        "The API is returning 500 errors",
        "We need a pricing proposal for enterprise tier",
        "Does this data processing comply with GDPR?"
    ]
    
    for i, req in enumerate(requests, 1):
        result = await rt.run_graph(
            graph=routing_graph,
            input_data=req,
            workflow_id=f"routing-request-{i:03d}"
        )
        print(f"\nRequest {i}: '{req[:50]}'")
        print(f"  workflow_id: {result.workflow_id}")
        print(f"  output:      {result.output}")
        print(f"  path_taken:  {result.execution_path}")
        print(f"  duration:    {result.duration_ms:.1f}ms")

asyncio.run(demo_run_graph())

## Section 6: rt.run_state_machine()

Run a `StateMachine` through the runtime. The runtime tracks the full trajectory (path) as a single workflow execution, including all intermediate state transitions.

In [ ]:
# --- rt.run_state_machine() ---

from multigen import StateMachine, Sampler

# Build a simple 3-state machine
async def draft_fn(inp, context=None):
    await asyncio.sleep(0.01)
    import random
    return {"content": f"Draft of: {inp}", "quality": round(random.uniform(0.5, 0.9), 2), "stage": "draft"}

async def review_fn(draft, context=None):
    await asyncio.sleep(0.01)
    import random
    prev = draft.get("quality", 0.6) if isinstance(draft, dict) else 0.6
    return {"content": "Reviewed: " + str(draft.get("content", "")),
            "quality": min(0.99, prev + random.uniform(0.05, 0.15)), "stage": "review"}

async def final_fn(reviewed, context=None):
    await asyncio.sleep(0.01)
    return {"content": "Published: " + str(reviewed.get("content", "")),
            "quality": reviewed.get("quality", 0.8), "stage": "final", "published": True}

runtime_sm = StateMachine(
    name="RuntimeWritingMachine",
    initial_state="draft",
    max_steps=10,
    sampler=Sampler.WEIGHTED,
    temperature=1.0
)
runtime_sm.state("draft",  FunctionAgent(draft_fn,  name="DraftAgent"))
runtime_sm.state("review", FunctionAgent(review_fn, name="ReviewAgent"))
runtime_sm.state("final",  FunctionAgent(final_fn,  name="FinalAgent"))
runtime_sm.transition("draft",  "review", prob=0.8)
runtime_sm.transition("draft",  "draft",  prob=0.2)
runtime_sm.transition("review", "final",  prob=0.9)
runtime_sm.transition("review", "draft",  prob=0.1)

async def demo_run_state_machine():
    print("=" * 60)
    print("rt.run_state_machine() — MCMC writing machine")
    print("=" * 60)
    
    result = await rt.run_state_machine(
        state_machine=runtime_sm,
        input_data="Write a blog post about Runtime observability.",
        workflow_id="writing-pipeline-mcmc-001"
    )
    
    print(f"workflow_id:    {result.workflow_id}")
    print(f"status:         {result.status}")
    print(f"duration_ms:    {result.duration_ms:.1f}ms")
    print(f"path:           {result.output.path}")
    print(f"steps:          {result.output.steps}")
    print(f"final_output:   {result.output.final_output}")

asyncio.run(demo_run_state_machine())

## Section 7: rt.stats()

After running workflows, `rt.stats()` returns aggregate statistics across all executions.

Stats include:
- Total runs, success rate, error count
- Average, p50, p95, p99 latency
- Breakdown by workflow type (chain, parallel, graph, state_machine)
- Breakdown by workflow_id prefix (if you use naming conventions)

In [ ]:
# --- rt.stats() after multiple runs ---

async def run_many_for_stats():
    """Run a bunch of workflows to generate meaningful stats."""
    # Run chains
    for i in range(5):
        await rt.run_chain(
            analysis_chain, f"Query {i}",
            workflow_id=f"chain-batch-{i:03d}"
        )
    
    # Run parallels
    for i in range(3):
        await rt.run_parallel(
            text_analysis_parallel, f"Text sample {i}",
            workflow_id=f"parallel-batch-{i:03d}"
        )
    
    # Run graphs
    for i in range(4):
        await rt.run_graph(
            routing_graph, f"Request {i}",
            workflow_id=f"graph-batch-{i:03d}"
        )

asyncio.run(run_many_for_stats())

# Get statistics
stats = rt.stats()

print("=" * 60)
print("RUNTIME STATISTICS")
print("=" * 60)
print()

# Overall counts
print("OVERALL:")
print(f"  total_runs:     {stats['total_runs']}")
print(f"  successful:     {stats['successful']}")
print(f"  failed:         {stats['failed']}")
print(f"  success_rate:   {stats['success_rate']:.1%}")
print()

# Latency stats
print("LATENCY (ms):")
print(f"  avg:   {stats['avg_latency_ms']:.1f}ms")
print(f"  p50:   {stats['p50_latency_ms']:.1f}ms")
print(f"  p95:   {stats['p95_latency_ms']:.1f}ms")
print(f"  p99:   {stats['p99_latency_ms']:.1f}ms")
print(f"  max:   {stats['max_latency_ms']:.1f}ms")
print()

# Breakdown by type
print("BREAKDOWN BY TYPE:")
for wf_type, type_stats in stats.get('by_type', {}).items():
    print(f"  {wf_type:15s}: {type_stats['count']:3d} runs, avg={type_stats['avg_ms']:.1f}ms, success={type_stats['success_rate']:.0%}")

## Section 8: get_runtime() — Process-Level Singleton

`get_runtime()` returns the same `Runtime` instance every time it's called within a process.

This is the **recommended pattern for production code**: configure the runtime once at application startup, then call `get_runtime()` anywhere in your codebase to execute workflows without passing the runtime object around.

In [ ]:
# --- get_runtime() singleton pattern ---

# In your application startup code (e.g., main.py or app.py):
# Configure the runtime once
configured_runtime = Runtime(
    simulator_url="http://localhost:8080",
    max_concurrency=10,
    trace=True
)

# Register it as the process singleton
# (In SDK v2+, this might be done via a context manager or set_runtime() call)
# For now, get_runtime() returns the default-configured singleton:

# From module A (e.g., agents/analysis.py)
rt_a = get_runtime()

# From module B (e.g., agents/reporting.py) — completely different file
rt_b = get_runtime()

# From module C (e.g., workflows/pipeline.py)
rt_c = get_runtime()

print("Singleton pattern demonstration:")
print(f"rt_a is rt_b: {rt_a is rt_b}")
print(f"rt_b is rt_c: {rt_b is rt_c}")
print(f"rt_a is rt_c: {rt_a is rt_c}")
print()
print("All three references point to the same Runtime instance.")
print("Stats accumulate across all callers — one unified view of execution.")

print()
print("Typical usage pattern:")
print()
print("# In main.py / startup:")
print("#   rt = Runtime(simulator_url='http://localhost:8080', trace=True)")
print("#   # Store as singleton somehow (SDK provides set_default_runtime() in v2+)")
print()
print("# In any module:")
print("#   rt = get_runtime()")
print("#   result = await rt.run_chain(my_chain, input_data)")

# Run something through the singleton
async def demo_singleton():
    rt_singleton = get_runtime()
    result = await rt_singleton.run_chain(
        analysis_chain,
        "Singleton pattern demo query",
        workflow_id="singleton-demo-001"
    )
    print(f"\nRan via singleton runtime: workflow_id={result.workflow_id}, status={result.status}")

asyncio.run(demo_singleton())

## Section 9: Simulator Integration

The Simulator is a **local web server** that provides a real-time visual dashboard of your workflow executions.

When you create a Runtime with `simulator_url="http://localhost:8080"`, every event generated during workflow execution is **POSTed** to the simulator's events endpoint.

### Starting the Simulator

```bash
# In a terminal:
python -m multigen.simulator
# → Simulator running at http://localhost:8080
```

### What Events Get Pushed

| Event Type | When It's Emitted |
|---|---|
| `workflow.started` | When `rt.run_*()` is called |
| `agent.started` | When an agent begins executing |
| `agent.completed` | When an agent finishes |
| `agent.failed` | When an agent raises an exception |
| `state.transition` | (StateMachine) When a state transition occurs |
| `workflow.completed` | When the full workflow finishes |
| `workflow.failed` | When the workflow fails |
| `custom` | User-defined events pushed manually |

### The Events Endpoint

Events are sent as HTTP POST to `{simulator_url}/api/v1/events` with a JSON payload.

In [ ]:
# --- Simulator integration explanation ---

# When the simulator is running, create a runtime that streams to it:
# rt_sim = Runtime(simulator_url="http://localhost:8080")
# result = await rt_sim.run_chain(my_chain, input_data)
# → You'll see all events appear in the dashboard at http://localhost:8080

# Let's show the exact event payload structure that gets sent to the simulator.
# We'll manually construct what an event looks like:

import json

# Example: what gets POSTed to http://localhost:8080/api/v1/events
sample_events = [
    {
        "event_type": "workflow.started",
        "level": "INFO",
        "trace_id": "wf-q4-analysis-001",
        "timestamp": 1735000000.123,
        "sender": "Runtime",
        "content": {
            "workflow_id": "q4-analysis-001",
            "workflow_type": "chain",
            "chain_name": "AnalysisChain",
            "input_preview": "Q4 sales analysis..."
        },
        "metadata": {
            "runtime_version": "2.0.0",
            "python_version": "3.11.0"
        }
    },
    {
        "event_type": "agent.started",
        "level": "DEBUG",
        "trace_id": "wf-q4-analysis-001",
        "timestamp": 1735000000.125,
        "sender": "FetchAgent",
        "content": {
            "agent_name": "FetchAgent",
            "position_in_chain": 0,
            "input_type": "str"
        },
        "metadata": {"workflow_id": "q4-analysis-001"}
    },
    {
        "event_type": "agent.completed",
        "level": "DEBUG",
        "trace_id": "wf-q4-analysis-001",
        "timestamp": 1735000000.148,
        "sender": "FetchAgent",
        "content": {
            "agent_name": "FetchAgent",
            "duration_ms": 23.1,
            "output_type": "dict",
            "success": True
        },
        "metadata": {"workflow_id": "q4-analysis-001"}
    },
    {
        "event_type": "workflow.completed",
        "level": "INFO",
        "trace_id": "wf-q4-analysis-001",
        "timestamp": 1735000000.200,
        "sender": "Runtime",
        "content": {
            "workflow_id": "q4-analysis-001",
            "total_duration_ms": 77.2,
            "agents_executed": 3,
            "status": "completed"
        },
        "metadata": {"workflow_id": "q4-analysis-001"}
    }
]

print("=" * 60)
print("SIMULATOR EVENT PAYLOADS")
print("=" * 60)
print("These JSON events are POSTed to: http://localhost:8080/api/v1/events")
print()

for i, event in enumerate(sample_events, 1):
    print(f"Event {i}: {event['event_type']}")
    print(json.dumps(event, indent=2))
    print()

## Section 10: Event Payload Structure

Every event has the same top-level structure:

```json
{
  "event_type": "agent.completed",   // Type of event
  "level":      "DEBUG",              // Log level: DEBUG/INFO/WARNING/ERROR
  "trace_id":   "wf-xyz-001",         // Groups all events from one workflow run
  "timestamp":  1735000000.148,       // Unix epoch float
  "sender":     "FetchAgent",         // Who emitted this event
  "content":    { ... },              // Event-specific data
  "metadata":   { ... }               // Extra context (workflow_id, tags, etc.)
}
```

The `trace_id` is the key field that ties all events from a single workflow execution together — essential for distributed tracing.

In [ ]:
# --- Event payload field reference ---

# Demonstrate which fields appear in which event types
print("=" * 70)
print("EVENT TYPE REFERENCE")
print("=" * 70)

event_types = {
    "workflow.started": {
        "content_fields": ["workflow_id", "workflow_type", "chain_name", "input_preview"],
        "level": "INFO",
        "description": "Emitted when rt.run_*() is called"
    },
    "agent.started": {
        "content_fields": ["agent_name", "position_in_chain", "input_type"],
        "level": "DEBUG",
        "description": "Emitted when an individual agent begins executing"
    },
    "agent.completed": {
        "content_fields": ["agent_name", "duration_ms", "output_type", "success"],
        "level": "DEBUG",
        "description": "Emitted when an agent finishes successfully"
    },
    "agent.failed": {
        "content_fields": ["agent_name", "error_type", "error_message", "traceback"],
        "level": "ERROR",
        "description": "Emitted when an agent raises an exception"
    },
    "state.transition": {
        "content_fields": ["from_state", "to_state", "probability", "step_count"],
        "level": "DEBUG",
        "description": "Emitted by StateMachine on each state transition"
    },
    "workflow.completed": {
        "content_fields": ["workflow_id", "total_duration_ms", "agents_executed", "status"],
        "level": "INFO",
        "description": "Emitted when the full workflow finishes"
    },
    "workflow.failed": {
        "content_fields": ["workflow_id", "error_type", "error_message", "failed_at_agent"],
        "level": "ERROR",
        "description": "Emitted when a workflow fails"
    },
    "custom": {
        "content_fields": ["any user-defined fields"],
        "level": "any",
        "description": "User-pushed custom events (see Section 11)"
    }
}

for event_type, info in event_types.items():
    print(f"\n{event_type}")
    print(f"  level:          {info['level']}")
    print(f"  description:    {info['description']}")
    print(f"  content fields: {info['content_fields']}")

## Section 11: Custom Event Pushing

You can push your own custom events to the simulator from **any code** — not just agents.

This is useful for:
- Marking significant milestones in your workflow ("batch complete", "model loaded")
- Pushing application-level metrics alongside agent events
- Debugging specific logic branches

In [ ]:
# --- Custom event pushing ---

async def demo_custom_events():
    """
    Push custom events to the simulator from non-agent code.
    Useful for marking milestones, adding context, or debugging.
    """
    print("=" * 60)
    print("CUSTOM EVENT PUSHING")
    print("=" * 60)
    
    # Start a workflow with a specific trace_id
    workflow_id = "custom-events-demo-001"
    
    # Push a custom 'batch.started' event before running the workflow
    await rt.push_event(
        event_type="custom",
        level="INFO",
        sender="BatchCoordinator",
        content={
            "message": "Starting batch processing run",
            "batch_id": workflow_id,
            "items_in_batch": 100,
            "estimated_duration_s": 45
        },
        trace_id=workflow_id,
        metadata={"environment": "production", "region": "us-east-1"}
    )
    print("Pushed custom 'batch.started' event")
    
    # Run the actual workflow
    result = await rt.run_chain(
        analysis_chain,
        "Batch item 1 of 100",
        workflow_id=workflow_id
    )
    print(f"Workflow completed: {result.status}")
    
    # Push a custom 'batch.checkpoint' event mid-workflow
    await rt.push_event(
        event_type="custom",
        level="DEBUG",
        sender="BatchCoordinator",
        content={
            "message": "Checkpoint: 1/100 items processed",
            "items_done": 1,
            "items_remaining": 99,
            "success_rate": 1.0
        },
        trace_id=workflow_id
    )
    print("Pushed custom 'checkpoint' event")
    
    # Push a final summary event
    await rt.push_event(
        event_type="custom",
        level="INFO",
        sender="BatchCoordinator",
        content={
            "message": "Batch demo complete",
            "workflow_result": result.status
        },
        trace_id=workflow_id
    )
    print("Pushed custom 'batch.complete' event")
    
    print()
    print("All custom events share trace_id with the workflow events.")
    print("In the Simulator UI, they appear in the same timeline.")

asyncio.run(demo_custom_events())

## Section 12: Temporal Backend (Conceptual)

**Temporal** is a durable workflow platform. When you use `rt.use_temporal(...)`, the Runtime routes workflow executions through Temporal instead of running them locally.

### What Temporal Provides

- **Durability**: Workflows survive process restarts, crashes, network failures
- **Long-running workflows**: Workflows can sleep for days/weeks, waiting for events
- **Automatic retry**: Temporal handles retry logic at the infrastructure level
- **Visibility**: Full workflow history in the Temporal UI
- **Replay**: Re-execute any workflow from history for debugging

### When to Use Temporal

- Workflows that take hours or days (e.g., human review pipelines)
- Workflows that must survive application restarts
- Workflows that need guaranteed exactly-once execution

In [ ]:
# --- Temporal backend (conceptual demonstration) ---
# NOTE: This requires a running Temporal server.
# Install: pip install temporalio
# Start server: docker run --rm temporalio/auto-setup:1.23

print("=" * 60)
print("TEMPORAL BACKEND (CONCEPTUAL)")
print("=" * 60)
print()
print("To use Temporal as the execution backend:")
print()
print("""
# Step 1: Create a Runtime with Temporal backend
rt_temporal = Runtime(
    simulator_url="http://localhost:8080",  # Still stream events to simulator
    max_concurrency=50
)

# Step 2: Configure the Temporal backend
await rt_temporal.use_temporal(
    server_address="localhost:7233",   # Temporal gRPC endpoint
    namespace="multigen-prod",          # Temporal namespace
    task_queue="ai-workflows",          # Worker task queue name
)

# Step 3: Run workflows exactly the same way
# The difference is NOW the workflow runs inside Temporal — durable, resumable
result = await rt_temporal.run_chain(
    analysis_chain,
    input_data="Long-running analysis",
    workflow_id="durable-analysis-001"
)
# workflow_id maps to a Temporal workflow ID
# If the process crashes mid-run, Temporal restarts from where it left off
""")

print()
print("Your workflow code (Chain, agents) is IDENTICAL whether running")
print("locally or on Temporal. The Runtime handles the translation.")
print()
print("Temporal workflow execution flow:")
print("  1. rt.run_chain() → Temporal submits workflow to task queue")
print("  2. Temporal worker picks up the workflow")
print("  3. Each agent.run() becomes a Temporal Activity")
print("  4. Activity results are persisted in Temporal's event history")
print("  5. If worker crashes: Temporal replays history to resume")
print("  6. Final result returned to caller")

## Section 13: Kafka Backend (Conceptual)

**Kafka** is a distributed streaming platform. When using `rt.use_kafka(...)`, workflow executions are submitted as Kafka messages and consumed by worker processes.

### What Kafka Provides

- **Distributed processing**: Many worker machines process workflows in parallel
- **Event sourcing**: Complete audit trail of all workflow events in Kafka topics
- **Backpressure**: Kafka buffers work when consumers are temporarily slow
- **Replay**: Replay historical workflow submissions by seeking to older offsets
- **Scale**: Easily scale to millions of workflow executions per day

In [ ]:
# --- Kafka backend (conceptual demonstration) ---
# NOTE: This requires a running Kafka broker.
# Install: pip install aiokafka
# Start: docker run --rm -p 9092:9092 apache/kafka:3.7.0

print("=" * 60)
print("KAFKA BACKEND (CONCEPTUAL)")
print("=" * 60)
print()
print("""
# Step 1: Create Runtime with Kafka backend
rt_kafka = Runtime(
    simulator_url="http://localhost:8080",
    max_concurrency=100  # Many concurrent consumers
)

# Step 2: Configure Kafka
await rt_kafka.use_kafka(
    bootstrap_servers="kafka:9092",    # Kafka broker address(es)
    topics={
        "workflows": "multigen.workflows",   # Workflow submission topic
        "results":   "multigen.results",     # Result delivery topic
        "events":    "multigen.events",      # Event stream topic
    },
    consumer_group="multigen-workers",   # Consumer group for load balancing
)

# Step 3: Submit workflow (non-blocking — returns immediately if async=True)
result = await rt_kafka.run_chain(
    analysis_chain,
    input_data="Distributed analysis",
    workflow_id="kafka-analysis-001",
    async_submit=True  # Returns a future, result comes via Kafka results topic
)
""")

print()
print("Kafka workflow execution flow:")
print("  1. rt.run_chain() → serializes workflow + input to Kafka message")
print("  2. Message is published to 'multigen.workflows' topic")
print("  3. One of many worker processes consumes the message")
print("  4. Worker runs the Chain locally and emits events to 'multigen.events' topic")
print("  5. Final result published to 'multigen.results' topic")
print("  6. Caller (or any subscriber) reads the result")
print()
print("All events also stream to the Simulator UI in real-time.")
print("This enables a full visual trace of distributed workflow execution.")

## Section 14: Workflow IDs and Tracing

Understanding how `workflow_id` and `trace_id` work together is essential for debugging complex multi-workflow systems.

### workflow_id

A human-readable identifier you provide (or auto-generated UUID). Identifies a **specific execution run**.

Good naming conventions:
- `"customer-{customer_id}-analysis-{date}"` — customer-specific workflows
- `"batch-{batch_id}-item-{n}"` — batch processing
- `"pipeline-{pipeline_name}-run-{run_number}"` — repeating pipelines

### trace_id

The `trace_id` groups ALL events from one workflow execution. It equals the `workflow_id` by default but can be overridden.

In distributed systems, the `trace_id` propagates across service boundaries — if workflow A spawns workflow B, both share the same `trace_id`.

In [ ]:
# --- Workflow IDs and tracing demonstration ---

async def demo_tracing():
    print("=" * 60)
    print("WORKFLOW IDs AND TRACING")
    print("=" * 60)
    
    # Run 3 workflows with meaningful IDs
    workflow_ids = [
        "customer-42-monthly-report-2025-12",
        "batch-etl-run-0847-item-001",
        "pipeline-news-analysis-run-099"
    ]
    
    results = []
    for wf_id in workflow_ids:
        result = await rt.run_chain(
            analysis_chain,
            f"Input for {wf_id}",
            workflow_id=wf_id
        )
        results.append(result)
        print(f"  Ran: {result.workflow_id}")
        print(f"       trace_id={result.trace_id}, duration={result.duration_ms:.1f}ms, status={result.status}")
    
    print()
    print("All events from 'customer-42-monthly-report-2025-12' share trace_id:")
    print(f"  {results[0].trace_id}")
    print()
    print("In the Simulator:")
    print("  Filter by trace_id → see only events from that workflow")
    print("  Timeline view → see all agent durations in a Gantt chart")
    print("  Diff view → compare two workflow runs side-by-side")
    print()
    print("In Jaeger/Zipkin (if trace=True):")
    print("  trace_id maps to an OpenTelemetry trace")
    print("  Each agent.run() is a span within the trace")
    print("  Spans have parent-child relationships (chain = sequential spans)")

asyncio.run(demo_tracing())

## Section 15: Real-World Example — 4-Agent Analysis Pipeline

Let's run a complete 4-agent analysis pipeline through the Runtime, showing all emitted events and stats.

The pipeline:
```
IngestAgent → CleanAgent → AnalyzeAgent → ReportAgent
```

We'll capture all events manually to show what the simulator would receive.

In [ ]:
# --- Full 4-agent analysis pipeline via Runtime ---

import random

# --- Define the 4 pipeline agents ---

async def ingest_fn(source_config, context=None):
    """Stage 1: Ingests raw data from configured sources."""
    await asyncio.sleep(0.03)
    return {
        "source": source_config.get("source", "unknown") if isinstance(source_config, dict) else "api",
        "records": random.randint(500, 10000),
        "raw_size_mb": round(random.uniform(1.0, 50.0), 1),
        "stage": "ingest"
    }

async def clean_fn(raw_data, context=None):
    """Stage 2: Cleans and validates the ingested data."""
    await asyncio.sleep(0.04)
    records = raw_data.get("records", 0) if isinstance(raw_data, dict) else 1000
    cleaned = int(records * random.uniform(0.85, 0.99))
    return {
        "clean_records": cleaned,
        "dropped_records": records - cleaned,
        "drop_rate": round((records - cleaned) / records, 3) if records > 0 else 0,
        "validated": True,
        "stage": "clean"
    }

async def analyze_fn(clean_data, context=None):
    """Stage 3: Runs statistical analysis on the cleaned data."""
    await asyncio.sleep(0.05)
    records = clean_data.get("clean_records", 0) if isinstance(clean_data, dict) else 900
    return {
        "records_analyzed": records,
        "mean": round(random.uniform(10, 1000), 2),
        "std":  round(random.uniform(1, 100), 2),
        "anomalies_detected": random.randint(0, 20),
        "confidence": round(random.uniform(0.85, 0.99), 3),
        "stage": "analyze"
    }

async def report_fn(analysis, context=None):
    """Stage 4: Generates a formatted report from the analysis."""
    await asyncio.sleep(0.02)
    anomalies = analysis.get("anomalies_detected", 0) if isinstance(analysis, dict) else 0
    severity = "HIGH" if anomalies > 10 else ("MEDIUM" if anomalies > 5 else "LOW")
    return {
        "report_id": f"RPT-{random.randint(10000, 99999)}",
        "anomaly_severity": severity,
        "summary": f"Analysis complete. {anomalies} anomalies detected. Severity: {severity}.",
        "recommendations": ["Investigate top anomalies", "Schedule follow-up review"],
        "format": "PDF",
        "stage": "report"
    }

# Build the 4-agent chain
full_pipeline = Chain(
    FunctionAgent(ingest_fn,  name="IngestAgent"),
    FunctionAgent(clean_fn,   name="CleanAgent"),
    FunctionAgent(analyze_fn, name="AnalyzeAgent"),
    FunctionAgent(report_fn,  name="ReportAgent"),
    name="DataAnalysisPipeline"
)

print("4-agent pipeline built: Ingest → Clean → Analyze → Report")

# Create a fresh runtime and run 3 different analysis jobs
rt_demo = Runtime()

async def run_full_demo():
    print()
    print("=" * 70)
    print("RUNNING 3 ANALYSIS JOBS VIA RUNTIME")
    print("=" * 70)
    
    jobs = [
        {"source": "database", "table": "transactions",   "date": "2025-12-01"},
        {"source": "api",      "endpoint": "/metrics",    "date": "2025-12-01"},
        {"source": "s3",       "bucket": "raw-events",    "date": "2025-12-01"}
    ]
    
    all_results = []
    for i, job in enumerate(jobs, 1):
        wf_id = f"analysis-job-{i:03d}-{job['source']}"
        print(f"\nJob {i}: source={job['source']}, workflow_id={wf_id}")
        
        result = await rt_demo.run_chain(
            full_pipeline,
            job,
            workflow_id=wf_id
        )
        all_results.append(result)
        
        # Show the pipeline output for each job
        print(f"  Status:      {result.status}")
        print(f"  Duration:    {result.duration_ms:.1f}ms")
        if isinstance(result.output, dict):
            print(f"  report_id:   {result.output.get('report_id')}")
            print(f"  severity:    {result.output.get('anomaly_severity')}")
            print(f"  summary:     {result.output.get('summary')}")
    
    # Final stats across all 3 jobs
    stats = rt_demo.stats()
    print()
    print("=" * 70)
    print("RUNTIME STATS AFTER 3 JOBS")
    print("=" * 70)
    print(f"  total_runs:     {stats['total_runs']}")
    print(f"  successful:     {stats['successful']}")
    print(f"  failed:         {stats['failed']}")
    print(f"  success_rate:   {stats['success_rate']:.1%}")
    print(f"  avg_latency:    {stats['avg_latency_ms']:.1f}ms")
    print(f"  p95_latency:    {stats['p95_latency_ms']:.1f}ms")
    
    print()
    print("If simulator was running, ALL these events would appear in the dashboard:")
    print("  3 × workflow.started")
    print("  3 × 4 = 12 × agent.started + agent.completed")
    print("  3 × workflow.completed")
    print(f"  = {3 + 12*2 + 3} total events in the simulator timeline")

asyncio.run(run_full_demo())

## Summary and Key Takeaways

In this notebook we explored the `Runtime` execution layer that ties all Multigen workflow types together:

### Core Concepts Mastered

| Concept | Method |
|---|---|
| **Unified execution** | `rt.run_chain()`, `rt.run_parallel()`, `rt.run_graph()`, `rt.run_state_machine()` |
| **Stats tracking** | `rt.stats()` — success rate, latency percentiles, breakdown by type |
| **Singleton pattern** | `get_runtime()` — shared runtime across entire codebase |
| **Simulator streaming** | `Runtime(simulator_url=...)` — stream events to visual dashboard |
| **Event structure** | `event_type`, `trace_id`, `sender`, `content`, `metadata` |
| **Custom events** | `rt.push_event(...)` — emit your own events to the simulator |
| **Temporal backend** | `rt.use_temporal(...)` — durable, resumable workflow execution |
| **Kafka backend** | `rt.use_kafka(...)` — distributed workflow execution at scale |
| **Workflow tracing** | `workflow_id` maps to `trace_id` — groups all events from one run |

### Architecture Principle

The Runtime implements the **Separation of Concerns** principle:
- Your **workflow code** (Chain, agents) contains only business logic
- The **Runtime** handles cross-cutting concerns: observability, backends, stats
- Switching from local execution → Temporal → Kafka requires **zero changes** to your workflow code

### Next Steps

- **Notebook 19**: Resilience — Circuit Breakers, Retry, Memory, Human-in-the-Loop